In [ ]:
from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Choose "smoke" on the university server; "train_5pct" on your friend's GPU.
MODE = "train_5pct"
assert MODE in ("smoke", "train_5pct")

NUM_BINS = 15_000  # Based on the observed training max bin 14,989; validate both splits below.
BATCH_SIZE = 16 if MODE == "smoke" else 64
EPOCHS = 1 if MODE == "smoke" else 8
LEARNING_RATE = 1e-3

DATA_DIR = Path("stage_1_5pct") if Path("stage_1_5pct/train.parquet").exists() else Path(".")
OUTPUT_DIR = Path("cnn_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Mode:", MODE)
print("Data folder:", DATA_DIR.resolve())
if MODE == "train_5pct" and DEVICE.type != "cuda":
    warnings.warn("Full 5% training will be slow without an NVIDIA GPU. Prefer the RTX 4090 PC.")

In [ ]:
required_files = ["train.parquet", "validation.parquet", "class_mapping.json"]
for name in required_files:
    if not (DATA_DIR / name).exists():
        raise FileNotFoundError(f"Missing {DATA_DIR / name}. Upload the files or update DATA_DIR.")

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
validation_full_df = pd.read_parquet(DATA_DIR / "validation.parquet")
with open(DATA_DIR / "class_mapping.json", "r") as f:
    class_mapping = json.load(f)

training_molecules = set(train_df["inchikey14"].dropna())
validation_df = validation_full_df[
    validation_full_df["inchikey14"].isin(training_molecules)
].copy()

assert train_df["inchikey14"].isin(class_mapping).all(), "Training labels missing from class_mapping.json"
assert validation_df["inchikey14"].isin(class_mapping).all(), "Filtered validation labels missing from class mapping"
assert len(set(class_mapping.values())) == len(class_mapping), "Mapping class IDs are not unique"
assert set(class_mapping.values()) == set(range(len(class_mapping))), "Class IDs must be 0,1,...,N-1"

print("Training spectra:", len(train_df))
print("Original validation spectra:", len(validation_full_df))
print("Validation spectra of known molecules:", len(validation_df))
print("Number of output classes:", len(class_mapping))
print("Validation coverage:", f"{len(validation_df) / len(validation_full_df):.2%}")

# Check a possible source-row identifier overlap; a result of zero is reassuring,
# but does not prove there are no duplicate spectra.
if {"source_part", "row_id"}.issubset(train_df.columns) and {"source_part", "row_id"}.issubset(validation_df.columns):
    key_train = set(zip(train_df["source_part"].astype(str), train_df["row_id"].astype(str)))
    key_val = set(zip(validation_df["source_part"].astype(str), validation_df["row_id"].astype(str)))
    overlap = len(key_train & key_val)
    print("Overlapping (source_part, row_id) identifiers:", overlap)
    if overlap:
        warnings.warn("Possible train/validation record overlap; review the split before reporting results.")


In [ ]:
def largest_bin(series):
    maximum = -1
    for bins in series:
        if bins is None:
            continue
        arr = np.asarray(bins)
        if arr.size:
            maximum = max(maximum, int(arr.max()))
    return maximum

max_train_bin = largest_bin(train_df["bins"])
max_valid_bin = largest_bin(validation_df["bins"])
print("Max train bin:", max_train_bin)
print("Max validation bin:", max_valid_bin)
if max(max_train_bin, max_valid_bin) >= NUM_BINS:
    raise ValueError(f"Found bin >= {NUM_BINS}. Increase NUM_BINS consistently for all experiments.")
print("Input shape per example will be (channels=1, positions=", NUM_BINS, ")")


In [ ]:
class SpectrumDataset(Dataset):
    def __init__(self, dataframe, mapping, num_bins):
        self.bins = dataframe["bins"].tolist()
        self.intensities = dataframe["bin_intensities"].tolist()
        labels = dataframe["inchikey14"].map(mapping)
        if labels.isna().any():
            raise ValueError("Dataset has molecules absent from the class mapping")
        self.labels = labels.to_numpy(dtype=np.int64)
        self.num_bins = num_bins

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        indices = np.asarray(self.bins[index], dtype=np.int64).reshape(-1)
        intensities = np.asarray(self.intensities[index], dtype=np.float32).reshape(-1)
        if indices.size != intensities.size:
            raise ValueError(f"Mismatched bins and intensities for row {index}")
        if np.any(indices < 0) or np.any(indices >= self.num_bins):
            raise ValueError(f"Out-of-range bin for row {index}")
        if not np.isfinite(intensities).all():
            raise ValueError(f"Non-finite intensity for row {index}")
        vector = np.zeros(self.num_bins, dtype=np.float32)
        # When duplicate bin positions occur, retain their maximum intensity.
        np.maximum.at(vector, indices, intensities)
        return torch.from_numpy(vector).unsqueeze(0), torch.tensor(self.labels[index], dtype=torch.long)

# Inspect one example without ever materialising all 91,851 dense vectors.
example_ds = SpectrumDataset(train_df.iloc[:1], class_mapping, NUM_BINS)
x0, y0 = example_ds[0]
print("Input tensor:", x0.shape, "  Target class:", y0.item())
print("Number of nonzero bins:", int((x0 != 0).sum()))


In [ ]:
class SmallSpectraCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=11, stride=4, padding=5),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=9, stride=4, padding=4),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=7, stride=4, padding=3),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(16),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallSpectraCNN(n_classes=len(class_mapping)).to(DEVICE)
parameter_count = sum(p.numel() for p in model.parameters())
print(f"Parameters: {parameter_count:,}")
print("Note: most parameters are in the large molecule-class output layer.")
with torch.no_grad():
    dummy = torch.zeros(2, 1, NUM_BINS, device=DEVICE)
    print("Output shape for 2 spectra:", tuple(model(dummy).shape))


In [ ]:
if MODE == "smoke":
    # Prefer spectra with molecule IDs represented in both splits so that
    # the tiny smoke test has at least some validation examples.
    eligible = train_df[train_df["inchikey14"].isin(validation_df["inchikey14"])]
    smoke_train_df = eligible.sample(n=min(256, len(eligible)), random_state=SEED)
    smoke_valid_df = validation_df[
        validation_df["inchikey14"].isin(smoke_train_df["inchikey14"])
    ].head(128)
    if len(smoke_train_df) == 0 or len(smoke_valid_df) == 0:
        raise ValueError("Unable to form smoke test. Review class overlap.")
    active_train_df, active_valid_df = smoke_train_df, smoke_valid_df
else:
    active_train_df, active_valid_df = train_df, validation_df

train_ds = SpectrumDataset(active_train_df, class_mapping, NUM_BINS)
valid_ds = SpectrumDataset(active_valid_df, class_mapping, NUM_BINS)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))
print("Running on", len(train_ds), "training and", len(valid_ds), "validation spectra")
if MODE == "smoke":
    print("SMOKE TEST ONLY: accuracy from this deliberately selected tiny sample is not reportable.")


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def sync_gpu():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def run_epoch(loader, training):
    model.train(training)
    total_loss, correct1, correct5, seen = 0.0, 0, 0, 0
    for inputs, labels in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(inputs)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        correct1 += (logits.argmax(dim=1) == labels).sum().item()
        top_k = min(5, logits.shape[1])
        correct5 += (logits.topk(top_k, dim=1).indices == labels[:, None]).any(dim=1).sum().item()
        seen += batch_size
    return {"loss": total_loss / seen,
            "top1_accuracy": correct1 / seen,
            "top5_accuracy": correct5 / seen}

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

history = []
for epoch in range(1, EPOCHS + 1):
    sync_gpu()
    start = time.perf_counter()
    training_metrics = run_epoch(train_loader, training=True)
    sync_gpu()
    train_seconds = time.perf_counter() - start
    validation_metrics = run_epoch(valid_loader, training=False)
    record = {
        "epoch": epoch,
        "train_seconds": train_seconds,
        **{f"train_{k}": v for k, v in training_metrics.items()},
        **{f"valid_{k}": v for k, v in validation_metrics.items()},
    }
    history.append(record)
    print(f"Epoch {epoch}: train loss {record['train_loss']:.4f}, "
          f"valid top-1 {record['valid_top1_accuracy']:.3%}, "
          f"valid top-5 {record['valid_top5_accuracy']:.3%}, "
          f"train time {train_seconds:.1f}s")

history_df = pd.DataFrame(history)
peak_memory_gb = (torch.cuda.max_memory_allocated() / (1024 ** 3)) if DEVICE.type == "cuda" else None
print("Peak allocated CUDA memory (GiB):", peak_memory_gb)


In [ ]:
run_name = "cnn_smoke" if MODE == "smoke" else "cnn_5pct"
history_df.to_csv(OUTPUT_DIR / f"{run_name}_metrics.csv", index=False)

metadata = {
    "mode": MODE,
    "random_seed": SEED,
    "architecture": "SmallSpectraCNN",
    "n_classes": len(class_mapping),
    "num_bins": NUM_BINS,
    "train_spectra": len(active_train_df),
    "validation_spectra": len(active_valid_df),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "pytorch_version": torch.__version__,
    "device": str(DEVICE),
    "parameters": parameter_count,
    "peak_allocated_cuda_memory_gib": peak_memory_gb,
    "validation_definition": "Only molecules appearing in the training sample",
    "note": "SMOKE metrics are not scientific results" if MODE == "smoke" else "5% known-molecule classification",
}
with open(OUTPUT_DIR / f"{run_name}_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

torch.save(model.state_dict(), OUTPUT_DIR / f"{run_name}_weights.pt")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Training loss")
ax.plot(history_df["epoch"], history_df["valid_loss"], marker="o", label="Validation loss")
ax.set(xlabel="Epoch", ylabel="Cross-entropy loss", title="CNN training progress")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"{run_name}_loss.png", dpi=160)
plt.show()

print("Saved files in:", OUTPUT_DIR.resolve())
